# Tutorial 8 – Equity Screening & Stock Selection Case
## FINM3422 – Professional Equity Research Workflow

**Estimated time:** ~2 hours (group-based)

---

### Scenario

You are part of a junior **equity research team** at an asset management firm.

Management has asked your team to:

> **Screen a universe of NASDAQ-listed technology companies and recommend ONE stock**
> for deeper fundamental analysis later in the course.

This notebook is intentionally structured like a **professional research notebook**,
following the modelling discipline discussed in **Lecture 6**.

You will work **collaboratively** using GitHub feature branches and pull requests.

## 1. Environment & Imports (≈5 minutes)

Professional modelling always begins by making the computing environment explicit.

Why this matters:
- Reproducibility across machines
- Clear collaboration in teams
- Fewer hidden bugs caused by implicit state

In this tutorial, we use **yfinance**, a free and stable Python library that provides
programmatic access to Yahoo Finance data.

No API key is required, which removes setup friction during tutorials.

In [13]:
# If needed (run once): pip install yfinance in console or use %pip install yfinance in your notebook
%pip install yfinance
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

Note: you may need to restart the kernel to use updated packages.


## What is the yfinance API?

`yfinance` is a Python wrapper around Yahoo Finance endpoints.
It allows analysts to retrieve:

- historical price data (daily, weekly, monthly)
- company metadata
- financial statements (income statement, balance sheet, cash flow)

It is:
- ✅ free
- ✅ fast to learn
- ✅ widely used for research prototyping and teaching

⚠️ Important:
`yfinance` is **not an institutional-grade data source** (e.g. Bloomberg or Refinitiv),
but it is perfectly suitable for **learning professional modelling workflows**.

## 2. Define the Investment Universe (≈5 minutes)

Equity research always begins with a clearly defined **investment universe**.

We restrict attention to a curated list of **large‑cap NASDAQ technology companies**.

Using a fixed universe ensures:
- Comparability across tutorial teams
- Stable API behaviour in class
- Focus on modelling rather than data hunting

In [23]:
tickers = [
    "AAPL",  # Apple
    "MSFT",  # Microsoft
    "NVDA",  # Nvidia
    "ADBE",  # Adobe
    "META",  # Meta Platforms
    "CRM",   # Salesforce
    "ORCL",  # Oracle
    "NOW",   # ServiceNow
    "INTU",  # Intuit
    "AMD"    # Advanced Micro Devices
]

## 3. Price Data Ingestion – API → DataFrame (≈15 minutes)

In professional workflows, analysts **do not manually download CSV files**.
Instead, data is pulled programmatically via APIs.

The main function used by yfinance is:

    yf.download()

### Key parameters you must understand

- `tickers`
  A single ticker (string) or multiple tickers (list)

- `interval`
  Controls **frequency** of observations  
  Examples:
  - `"1d"`  → daily data
  - `"1wk"` → weekly data
  - `"1mo"` → monthly data

- `period`
  Controls **how much history** is returned  
  Examples:
  - `"1y"`   → last year
  - `"5y"`   → last 5 years
  - `"max"`  → full available history

⚠️ **Critical rule**
> Never rely on API defaults. Always specify `period` explicitly.

## Why We Use Monthly Data in This Tutorial

In FINM3422:
- portfolio returns are monthly
- performance metrics are monthly
- Assignment 2 uses monthly frequency

Therefore, we deliberately request:
- `interval = "1mo"` → monthly observations
- `period = "max"`   → long historical window

This ensures:
- frequency is correct before computing returns
- enough observations exist for rolling metrics
- alignment with Lecture 6 modelling discipline

In [55]:
# TODO:
# Use yf.download to retrieve monthly adjusted prices
# - interval="1mo"
# - period="max"
# - auto_adjust=True

import yfinance as yf
import pandas as pd

prices = yf.download(
    tickers,
    period="max",
    interval="1mo",
    auto_adjust=True,
    progress=False
)

prices = prices["Close"] if isinstance(prices.columns, pd.MultiIndex) else prices[["Close"]]
prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()

print(prices.head())



# Then:
# - keep adjusted close prices
# - ensure DatetimeIndex
# - sort index

prices = yf.download(
    tickers,
    period="max",
    interval="1mo",
    auto_adjust=True,
    progress=False
)

prices = prices["Close"] if isinstance(prices.columns, pd.MultiIndex) else prices[["Close"]]
prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()

Ticker          AAPL  ADBE      AMD  CRM  INTU  META  MSFT  NOW  NVDA  ORCL
Date                                                                       
1985-01-01  0.099152   NaN  17.5625  NaN   NaN   NaN   NaN  NaN   NaN   NaN
1985-02-01  0.084621   NaN  16.6250  NaN   NaN   NaN   NaN  NaN   NaN   NaN
1985-03-01  0.075646   NaN  16.6875  NaN   NaN   NaN   NaN  NaN   NaN   NaN
1985-04-01  0.072655   NaN  13.9375  NaN   NaN   NaN   NaN  NaN   NaN   NaN
1985-05-01  0.059406   NaN  13.2500  NaN   NaN   NaN   NaN  NaN   NaN   NaN


## 4. Data Inspection & Sanity Checks (≈10 minutes)

Before computing metrics, analysts **inspect the raw data**.

You should check:
- `.info()`
- missing values (`.isna().sum()`)
- index type (`type(df.index)`)
- chronological ordering

### Important modelling principle (Lecture 6)

> NA does **not** mean bad data.
> It often means the company did not exist yet (IPO timing).

Different firms have different listing dates.
This is **economically correct**, not an error.

In [71]:
# TODO:
# Inspect the price DataFrame:
# - info()
# - missing values
# - index properties

print("Price DataFrame info:")
prices.info()

print("\nMissing values by column:")
print(prices.isna().sum())

print("\nIndex properties:")
print("Type:", type(prices.index))
print("Is DatetimeIndex:", isinstance(prices.index, pd.DatetimeIndex))
print("Monotonic increasing:", prices.index.is_monotonic_increasing)
print("Has duplicates:", prices.index.has_duplicates)
print("Start:", prices.index.min())
print("End:", prices.index.max())

missing_pct = (prices.isna().mean() * 100).round(2)
print("\nMissing % by column:")
print(missing_pct)

Price DataFrame info:
<class 'pandas.DataFrame'>
DatetimeIndex: 496 entries, 1985-01-01 to 2026-04-01
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    496 non-null    float64
 1   ADBE    477 non-null    float64
 2   AMD     496 non-null    float64
 3   CRM     263 non-null    float64
 4   INTU    398 non-null    float64
 5   META    168 non-null    float64
 6   MSFT    482 non-null    float64
 7   NOW     167 non-null    float64
 8   NVDA    328 non-null    float64
 9   ORCL    482 non-null    float64
dtypes: float64(10)
memory usage: 42.6 KB

Missing values by column:
Ticker
AAPL      0
ADBE     19
AMD       0
CRM     233
INTU     98
META    328
MSFT     14
NOW     329
NVDA    168
ORCL     14
dtype: int64

Index properties:
Type: <class 'pandas.DatetimeIndex'>
Is DatetimeIndex: True
Monotonic increasing: True
Has duplicates: False
Start: 1985-01-01 00:00:00
End: 2026-04-01 00:00:00

Missing % by column:
Ticker
AAP

## 5. Returns & Performance Metrics (≈15 minutes)

Equity screeners often include **recent market performance**
(e.g. momentum or confirmation).

Tasks:
- Compute monthly returns
- Compute trailing 12‑month total return

⚠️ Always fix frequency *before* computing returns.

In [72]:
# TODO:
# 1. Compute monthly returns using pct_change()
# 2. Compute trailing 12‑month total return

## 6. Visual Diagnostics – Wealth Index (≈15 minutes)

Before trusting any metric, professional analysts **plot a wealth index**.

A wealth index answers:
> What happens to $1 invested over time?

This diagnostic reveals:
- missing months
- compounding errors
- extreme outliers dominating results

In [73]:
# TODO:
# Construct and plot wealth indices
# Plot at least 3 stocks
# Add a brief interpretation

## 7. Fundamental Indicators – Growth & Quality (≈20 minutes)

Market prices alone are not sufficient for equity analysis.

We complement price performance with **fundamentals**.

Two indicators:
- **Revenue growth** (latest year vs previous year)
- **Operating margin** (operating income / revenue)

Financial statements are accessed using:

    yf.Ticker("AAPL").financials

In [78]:
# TODO:
# For each ticker:
# - retrieve financial statements using yf.Ticker
# - compute revenue growth
# - compute operating margin
# Store results in a DataFrame

import pandas as pd
import yfinance as yf

results = []

for ticker in tickers:
    t = yf.Ticker(ticker)
    fin = t.financials

    if fin is None or fin.empty or fin.shape[1] < 2:
        results.append({
            "Ticker": ticker,
            "Revenue Growth": None,
            "Operating Margin": None
        })
        continue

    fin = fin.sort_index(axis=1, ascending=True)

    revenue = fin.loc["Total Revenue"] if "Total Revenue" in fin.index else None
    op_income = fin.loc["Operating Income"] if "Operating Income" in fin.index else None

    if revenue is not None and len(revenue.dropna()) >= 2:
        rev_growth = revenue.iloc[-1] / revenue.iloc[-2] - 1
    else:
        rev_growth = None

    if revenue is not None and op_income is not None and pd.notna(revenue.iloc[-1]) and pd.notna(op_income.iloc[-1]) and revenue.iloc[-1] != 0:
        op_margin = op_income.iloc[-1] / revenue.iloc[-1]
    else:
        op_margin = None

    results.append({
        "Ticker": ticker,
        "Revenue Growth": rev_growth,
        "Operating Margin": op_margin
    })

fundamentals_df = pd.DataFrame(results)

fundamentals_df["Revenue Growth"] = fundamentals_df["Revenue Growth"].map(lambda x: f"{x:.2%}" if pd.notna(x) else "N/A")
fundamentals_df["Operating Margin"] = fundamentals_df["Operating Margin"].map(lambda x: f"{x:.2%}" if pd.notna(x) else "N/A")

print(fundamentals_df.to_string(index=False))


Ticker Revenue Growth Operating Margin
  AAPL          6.43%           31.97%
  MSFT         14.93%           45.62%
  NVDA         65.47%           60.38%
  ADBE         10.53%           36.63%
  META         22.17%           41.44%
   CRM          9.58%           21.47%
  ORCL          8.38%           31.45%
   NOW         20.88%           13.74%
  INTU         15.63%           26.22%
   AMD         34.34%           10.66%


## 8. Screening Table Construction (≈15 minutes)

Professional screeners combine **all signals into one master table**.

Structure:
- each row → one stock
- each column → one screening signal

This table forms the basis for ranking and decision‑making.

In [83]:
# TODO:
# Combine:
# - 12‑month return
# - revenue growth
# - operating margin
# into a single screening table

screening_table = pd.DataFrame(results)

twelve_month_returns = (prices.iloc[-1] / prices.iloc[-13] - 1)

screening_table["12M Return"] = screening_table["Ticker"].map(twelve_month_returns)
screening_table["Revenue Growth"] = screening_table["Revenue Growth"].map(lambda x: f"{x:.2%}" if pd.notna(x) else "N/A")
screening_table["Operating Margin"] = screening_table["Operating Margin"].map(lambda x: f"{x:.2%}" if pd.notna(x) else "N/A")
screening_table["12M Return"] = screening_table["12M Return"].map(lambda x: f"{x:.2%}" if pd.notna(x) else "N/A")

screening_table = screening_table[["Ticker", "12M Return", "Revenue Growth", "Operating Margin"]]

print(screening_table.to_string(index=False))

Ticker 12M Return Revenue Growth Operating Margin
  AAPL     29.11%          6.43%           31.97%
  MSFT     10.37%         14.93%           45.62%
  NVDA     85.96%         65.47%           60.38%
  ADBE    -31.75%         10.53%           36.63%
  META     23.28%         22.17%           41.44%
   CRM    -28.73%          9.58%           21.47%
  ORCL     35.06%          8.38%           31.45%
   NOW    -46.04%         20.88%           13.74%
  INTU    -34.22%         15.63%           26.22%
   AMD    211.72%         34.34%           10.66%


## 9. Ranking & Shortlisting Logic (≈15 minutes)

Equity screening is about **narrowing the universe**, not precision.

Rank stocks using:
- revenue growth
- operating margin
- 12‑month performance

Combine ranks into a **composite score**.

There is no single correct answer —
only defensible logic that can be explained.

In [86]:
# TODO:
# Rank each metric
# Compute composite score
# Sort stocks from most to least attractive

import pandas as pd

screening = screening_table.copy()

# Convert percent strings back to numeric for ranking
for col in ["12M Return", "Revenue Growth", "Operating Margin"]:
    screening[col] = screening[col].replace("N/A", pd.NA)
    screening[col] = screening[col].astype(str).str.replace("%", "", regex=False).astype(float) / 100

# Rank each metric: higher is better
screening["Return Rank"] = screening["12M Return"].rank(ascending=False, method="min")
screening["Growth Rank"] = screening["Revenue Growth"].rank(ascending=False, method="min")
screening["Margin Rank"] = screening["Operating Margin"].rank(ascending=False, method="min")

# Composite score: average rank
screening["Composite Score"] = screening[["Return Rank", "Growth Rank", "Margin Rank"]].mean(axis=1)

# Sort from best to worst
screening = screening.sort_values("Composite Score").reset_index(drop=True)

# Format back to percentages for display
display_df = screening.copy()
for col in ["12M Return", "Revenue Growth", "Operating Margin"]:
    display_df[col] = display_df[col].map(lambda x: f"{x:.2%}" if pd.notna(x) else "N/A")

print(display_df[["Ticker", "12M Return", "Revenue Growth", "Operating Margin", "Composite Score"]].to_string(index=False))

Ticker 12M Return Revenue Growth Operating Margin  Composite Score
  NVDA     85.96%         65.47%           60.38%         1.333333
  META     23.28%         22.17%           41.44%         3.666667
   AMD    211.72%         34.34%           10.66%         4.333333
  MSFT     10.37%         14.93%           45.62%         4.666667
  ORCL     35.06%          8.38%           31.45%         6.000000
  ADBE    -31.75%         10.53%           36.63%         6.333333
  AAPL     29.11%          6.43%           31.97%         6.333333
  INTU    -34.22%         15.63%           26.22%         7.000000
   NOW    -46.04%         20.88%           13.74%         7.666667
   CRM    -28.73%          9.58%           21.47%         7.666667


## 10. Stock Selection – Analyst Decision (≈10 minutes)

Your team must now make a **judgement call**.

Select **ONE stock** to recommend for deeper analysis.

Write as if presenting to an **investment committee**.

In [87]:
### Selected Stock

### Analyst Rationale
- Growth outlook:
- Profitability / quality:
- Market performance:
- Key risks:

SyntaxError: invalid syntax (255545209.py, line 4)

## 11. GitHub Collaboration Reflection (≈10 minutes)

Equity research is a **team activity**.

Document:
- Who worked on which sections
- How feature branches and pull requests were used
- Any challenges encountered

- Member A:
- Member B:
- Member C: